In [32]:
import torch
torch.cuda.empty_cache()

In [33]:
torch.cuda.ipc_collect()

In [34]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [35]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [36]:
!kill -9 626

/bin/bash: line 1: kill: (626) - No such process


In [37]:
torch.cuda.empty_cache()

In [38]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from timm import create_model

import numpy as np
import cv2
from PIL import Image
import gc

from tqdm import tqdm
from torch.amp import autocast, GradScaler

In [39]:
class DualImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(root)
        self.transform = transform

    def apply_clahe(self, img):
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = clahe.apply(img)

        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(img)

    def __getitem__(self, index):
        path, label = self.samples[index]
        img = self.loader(path)

        img_clahe = self.apply_clahe(img)

        img1 = self.transform(img)
        img2 = self.transform(img_clahe)

        return img1, img2, label

In [40]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [41]:
data_dir = "/kaggle/input/datasets/venkatsaikondra/multi-venkat/Final_Data"

train_dataset = DualImageFolder(data_dir + "/train", transform=train_transform)
val_dataset = DualImageFolder(data_dir + "/val", transform=val_transform)
test_dataset = DualImageFolder(data_dir + "/test", transform=val_transform)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

In [42]:
class ResViT(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # CNN
        self.cnn = models.resnet18(weights="IMAGENET1K_V1")
        cnn_dim = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        # ViT
        self.vit = create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
        vit_dim = 384

        self.feature_dim = cnn_dim + vit_dim
        self.total_dim = self.feature_dim * 2

        # 🔥 Projection head (for contrastive)
        self.projection = nn.Sequential(
            nn.Linear(self.total_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.total_dim),
            nn.Linear(self.total_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def extract_features(self, x):
        f_cnn = self.cnn(x)
        f_vit = self.vit(x)

        f_cnn = nn.functional.normalize(f_cnn, dim=1)
        f_vit = nn.functional.normalize(f_vit, dim=1)

        return torch.cat([f_cnn, f_vit], dim=1)

    def forward(self, x1, x2):
        f1 = self.extract_features(x1)
        f2 = self.extract_features(x2)

        f = torch.cat([f1, f2], dim=1)

        logits = self.classifier(f)
        proj = self.projection(f)

        return logits, proj

In [43]:
def contrastive_loss(features, labels, temperature=0.5):
    features = nn.functional.normalize(features, dim=1)
    similarity = torch.matmul(features, features.T)

    labels = labels.unsqueeze(1)
    mask = torch.eq(labels, labels.T).float()

    exp_sim = torch.exp(similarity / temperature)
    loss = -torch.log(exp_sim / exp_sim.sum(dim=1, keepdim=True))

    return (mask * loss).mean()

In [44]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ResViT().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

scaler = GradScaler()

In [45]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    all_preds, all_labels = [], []

    with torch.no_grad():
        for img1, img2, labels in loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)

            outputs, _ = model(img1, img2)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(classification_report(all_labels, all_preds))
    print(confusion_matrix(all_labels, all_preds))

    return correct / total

In [46]:
def train_model(model, train_loader, val_loader, epochs=20):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for img1, img2, labels in tqdm(train_loader):
            img1 = img1.to(device)
            img2 = img2.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with autocast(device_type='cuda'):
                outputs, features = model(img1, img2)

                ce_loss = criterion(outputs, labels)
                cont_loss = contrastive_loss(features, labels)

                loss = ce_loss + 0.3 * cont_loss  # 🔥 combined loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        scheduler.step()

        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    return model

In [47]:
model = train_model(model, train_loader, val_loader, epochs=25)

print("\nFinal Test Performance:")
test_acc = evaluate(model, test_loader)
print(f"Final Test Accuracy: {test_acc:.4f}")

100%|██████████| 944/944 [01:47<00:00,  8.80it/s]


              precision    recall  f1-score   support

           0       0.98      1.00      0.99       404
           1       0.97      0.88      0.93       404
           2       0.85      0.62      0.72       404
           3       0.66      0.89      0.76       404

    accuracy                           0.85      1616
   macro avg       0.87      0.85      0.85      1616
weighted avg       0.87      0.85      0.85      1616

[[402   0   0   2]
 [  7 357   5  35]
 [  0   4 252 148]
 [  0   6  39 359]]
Epoch 1, Loss: 879.6808, Val Acc: 0.8478


100%|██████████| 944/944 [01:48<00:00,  8.67it/s]


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.88      0.98      0.93       404
           2       0.62      0.93      0.74       404
           3       0.93      0.37      0.53       404

    accuracy                           0.82      1616
   macro avg       0.86      0.82      0.80      1616
weighted avg       0.86      0.82      0.80      1616

[[402   0   1   1]
 [  1 394   9   0]
 [  0  19 375  10]
 [  1  34 220 149]]
Epoch 2, Loss: 745.7653, Val Acc: 0.8168


100%|██████████| 944/944 [01:50<00:00,  8.56it/s]


...

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.98      0.98      0.98       404
           2       0.79      0.93      0.85       404
           3       0.90      0.75      0.82       404

    accuracy                           0.91      1616
   macro avg       0.92      0.91      0.91      1616
weighted avg       0.92      0.91      0.91      1616

[[403   0   0   1]
 [  2 394   4   4]
 [  0   3 374  27]
 [  0   6  97 301]]
Epoch 23, Loss: 516.3197, Val Acc: 0.9109


100%|██████████| 944/944 [01:47<00:00,  8.81it/s]


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.97      0.99      0.98       404
           2       0.87      0.88      0.88       404
           3       0.88      0.86      0.87       404

    accuracy                           0.93      1616
   macro avg       0.93      0.93      0.93      1616
weighted avg       0.93      0.93      0.93      1616

[[403   0   0   1]
 [  1 398   2   3]
 [  0   5 354  45]
 [  0   7  49 348]]
Epoch 24, Loss: 507.1942, Val Acc: 0.9301


100%|██████████| 944/944 [01:46<00:00,  8.85it/s]


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.99      0.98      0.98       404
           2       0.83      0.91      0.87       404
           3       0.89      0.81      0.85       404

    accuracy                           0.92      1616
   macro avg       0.93      0.92      0.92      1616
weighted avg       0.93      0.92      0.92      1616

[[403   0   0   1]
 [  1 396   3   4]
 [  0   2 367  35]
 [  0   4  72 328]]
Epoch 25, Loss: 508.4103, Val Acc: 0.9245

Final Test Performance:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       405
           1       0.98      0.98      0.98       405
           2       0.80      0.90      0.84       405
           3       0.88      0.77      0.82       405

    accuracy                           0.91      1620
   macro avg       0.91      0.91      0.91      1620
weighted avg       0.91      0.91      0.